<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install SimpleITK==2.4.0 -q
%pip install numpy==1.26.4 -q
%pip install pyradiomics==3.1.0 -q
%pip install pandas==1.2.3 -q

In [3]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def featureExtractor(fileId):
  imagePath = './dataset/BraTS2021_Training_Data/%s/%s_flair.nii.gz' % (fileId, fileId)
  image = sitk.ReadImage(imagePath)
  maskPath ='./dataset/BraTS2021_Training_Data/%s/%s_kernel5_non_tumor.nii.gz' % (fileId, fileId)
  mask = sitk.ReadImage(maskPath)

  kernel = 5
  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 40000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName('glszm')

  featureMap = extractor.execute(image, mask, voxelBased=True)

  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      fileFolder = './dataset/glszm/kernel5-radius5/non_tumor/%s' % (fileId)
      if path.exists(fileFolder) == False:
        os.mkdir(fileFolder)
      sitk.WriteImage(featureValue, '%s/%s.nrrd' % (fileFolder, featureName))
      print('Computed %s, stored as "%s/%s.nrrd"' % (featureName, fileFolder, featureName))
    # else:
    #   print('%s: %s' % (featureName, featureValue))

monitorFilePath = './dataset/glszm/kernel5-radius5/non_tumor/glszm.monitor.csv'
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  fileId = row['file']
  print('Starting %s' % (fileId))
  featureExtractor(fileId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00094
Computed original_glszm_GrayLevelNonUniformity, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_GrayLevelNonUniformity.nrrd"
Computed original_glszm_GrayLevelNonUniformityNormalized, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_GrayLevelNonUniformityNormalized.nrrd"
Computed original_glszm_GrayLevelVariance, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_GrayLevelVariance.nrrd"
Computed original_glszm_HighGrayLevelZoneEmphasis, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_HighGrayLevelZoneEmphasis.nrrd"
Computed original_glszm_LargeAreaEmphasis, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_LargeAreaEmphasis.nrrd"
Computed original_glszm_LargeAreaHighGrayLevelEmphasis, stored as "./dataset/glszm/kernel5-radius5/non_tumor/BraTS2021_00094/original_glszm_LargeAreaHighGrayLevelEm